# Age & Gender Recognition: Face Detection + Attributes (UTKFace)Notebook-first refactor with real Kaggle UTKFace download, verified parsing/splits, face detection checks, custom age-gender head training, and saved artifacts.

## 1. Problem Type and Method FitTask family: face detection + attributes (age and gender).- Face detector: DeepFace-backed detector used for detection quality checks and optional qualitative inference.- Attribute model: custom multi-task PyTorch head (gender classification + age regression).- Evaluation: quantitative metrics for gender and age plus qualitative examples.

## 2. Dataset Source and License NotesDataset link: https://www.kaggle.com/datasets/jangedoo/utkface-newUse this dataset according to Kaggle terms and the dataset publisher's licensing notes.

## 3. Environment, Seed, and Device

In [ ]:
import platformimport randomimport numpy as npimport torchSEED = 42random.seed(SEED)np.random.seed(SEED)torch.manual_seed(SEED)if torch.cuda.is_available():    torch.cuda.manual_seed_all(SEED)DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'print(f'Python: {platform.python_version()}')print(f'PyTorch: {torch.__version__}')print(f'Device: {DEVICE}')if torch.cuda.is_available():    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 4. Install Dependencies

In [ ]:
import importlibimport subprocessimport sysdef ensure_package(import_name, pip_name=None):    pip_name = pip_name or import_name    try:        importlib.import_module(import_name)    except ImportError:        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])ensure_package('kagglehub')ensure_package('cv2', 'opencv-python')ensure_package('pandas')ensure_package('matplotlib')ensure_package('sklearn', 'scikit-learn')ensure_package('PIL', 'Pillow')ensure_package('deepface')ensure_package('torchvision')print('Dependencies are available.')

## 5. Imports and Paths

In [ ]:
import jsonimport osimport refrom pathlib import Pathimport cv2import matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltimport numpy as npimport pandas as pdimport torch.nn as nnfrom PIL import Imagefrom sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, mean_absolute_error, mean_squared_errorfrom sklearn.model_selection import train_test_splitfrom torch.utils.data import Dataset, DataLoaderfrom torchvision import models, transformsfrom deepface import DeepFacePROJECT_DIR = Path.cwd()WORK_DIR = PROJECT_DIR / 'working'DATA_DIR = WORK_DIR / 'data'ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'RUNS_DIR = PROJECT_DIR / 'runs'WORK_DIR.mkdir(parents=True, exist_ok=True)DATA_DIR.mkdir(parents=True, exist_ok=True)ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)RUNS_DIR.mkdir(parents=True, exist_ok=True)KAGGLE_DATASET = 'jangedoo/utkface-new'print(f'Project dir: {PROJECT_DIR}')

## 6. Real Dataset Download (UTKFace from Kaggle)

In [ ]:
def check_kaggle_credentials():    has_env = bool(os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))    has_file = (Path.home() / '.kaggle' / 'kaggle.json').exists()    if not has_env and not has_file:        raise RuntimeError('Missing Kaggle credentials. Set KAGGLE_USERNAME/KAGGLE_KEY or create ~/.kaggle/kaggle.json')def download_utkface():    check_kaggle_credentials()    import kagglehub    return Path(kagglehub.dataset_download(KAGGLE_DATASET))raw_root = Path(download_utkface())print(f'Raw dataset path: {raw_root}')

## 7. Build Metadata from Filename Labels and Validate Data

In [ ]:
image_paths = sorted(list(raw_root.rglob('*.jpg')) + list(raw_root.rglob('*.png')))if len(image_paths) == 0:    raise RuntimeError('No image files found after dataset download')pattern = re.compile(r'^(\d+)_(\d)_(\d)_(\d+)$')records = []bad_name_count = 0for p in image_paths:    stem = p.stem    m = pattern.match(stem)    if m is None:        bad_name_count += 1        continue    age = int(m.group(1))    gender = int(m.group(2))    race = int(m.group(3))    if age < 0 or age > 120:        continue    if gender not in [0, 1]:        continue    records.append({'path': str(p), 'age': age, 'gender': gender, 'race': race})df = pd.DataFrame(records)if df.empty:    raise RuntimeError('No valid UTKFace rows after filename parsing')missing_files = int((~df['path'].apply(lambda x: Path(x).exists())).sum())if missing_files > 0:    raise RuntimeError(f'Missing files referenced by metadata: {missing_files}')sample_open = df['path'].sample(n=min(200, len(df)), random_state=SEED).tolist()corrupted = 0for fp in sample_open:    img = cv2.imread(fp)    if img is None:        corrupted += 1if corrupted > 0:    raise RuntimeError(f'Corrupted images found in sample check: {corrupted}')df = df.drop_duplicates(subset=['path']).reset_index(drop=True)print(f'Valid rows: {len(df)}')print(f'Ignored malformed names: {bad_name_count}')print(f'Age range: {df.age.min()} to {df.age.max()}')print(f'Gender counts: {df.gender.value_counts().to_dict()}')

## 8. Split Verification (Train / Validation / Test)

In [ ]:
df['age_bin'] = pd.cut(df['age'], bins=[0, 12, 20, 35, 50, 70, 120], labels=False, include_lowest=True)df['strata'] = df['gender'].astype(str) + '_' + df['age_bin'].astype(str)train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df['strata'])val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['strata'])if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:    raise RuntimeError('One or more data splits are empty')print(f'Train: {len(train_df)}')print(f'Val:   {len(val_df)}')print(f'Test:  {len(test_df)}')print('Train gender:', train_df.gender.value_counts().to_dict())print('Val gender:  ', val_df.gender.value_counts().to_dict())print('Test gender: ', test_df.gender.value_counts().to_dict())

## 9. EDA: Samples and Distributions

In [ ]:
sample_df = train_df.sample(n=min(9, len(train_df)), random_state=SEED)fig, axes = plt.subplots(3, 3, figsize=(12, 12))for ax, (_, row) in zip(axes.flatten(), sample_df.iterrows()):    img = cv2.imread(row['path'])    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)    ax.imshow(img)    ax.set_title(f"age={row['age']}, gender={row['gender']}")    ax.axis('off')for ax in axes.flatten()[len(sample_df):]:    ax.axis('off')plt.tight_layout()sample_grid_path = ARTIFACTS_DIR / 'eda_samples.png'plt.savefig(sample_grid_path, dpi=150, bbox_inches='tight')plt.show()fig, ax = plt.subplots(1, 2, figsize=(12, 4))ax[0].hist(train_df['age'], bins=30, color='steelblue', edgecolor='white')ax[0].set_title('Train Age Distribution')ax[1].bar(['male(0)', 'female(1)'], [int((train_df.gender==0).sum()), int((train_df.gender==1).sum())], color=['teal', 'coral'])ax[1].set_title('Train Gender Distribution')plt.tight_layout()eda_dist_path = ARTIFACTS_DIR / 'eda_distributions.png'plt.savefig(eda_dist_path, dpi=150, bbox_inches='tight')plt.show()print(f'Saved EDA artifacts: {sample_grid_path}, {eda_dist_path}')

## 10. Face Detection Quality Check (DeepFace Detector)

In [ ]:
detect_sample = val_df.sample(n=min(80, len(val_df)), random_state=SEED)['path'].tolist()detected = 0for fp in detect_sample:    try:        faces = DeepFace.extract_faces(img_path=fp, detector_backend='opencv', enforce_detection=False, align=True)        if len(faces) > 0:            detected += 1    except Exception:        continuedetection_rate = detected / max(1, len(detect_sample))print(f'Detection sample size: {len(detect_sample)}')print(f'Detected faces: {detected}')print(f'Detection rate: {detection_rate:.4f}')

## 11. Dataset Class and DataLoaders for Custom Head

In [ ]:
IMG_SIZE = 160BATCH_SIZE = 32 if DEVICE == 'cuda' else 16train_tfms = transforms.Compose([    transforms.Resize((IMG_SIZE, IMG_SIZE)),    transforms.RandomHorizontalFlip(p=0.5),    transforms.ColorJitter(brightness=0.2, contrast=0.2),    transforms.ToTensor(),    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])eval_tfms = transforms.Compose([    transforms.Resize((IMG_SIZE, IMG_SIZE)),    transforms.ToTensor(),    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])class AgeGenderDataset(Dataset):    def __init__(self, frame, tfms):        self.frame = frame.reset_index(drop=True)        self.tfms = tfms    def __len__(self):        return len(self.frame)    def __getitem__(self, idx):        row = self.frame.iloc[idx]        img = Image.open(row['path']).convert('RGB')        x = self.tfms(img)        age = torch.tensor(float(row['age']), dtype=torch.float32)        gender = torch.tensor(int(row['gender']), dtype=torch.long)        return x, age, gender, row['path']train_ds = AgeGenderDataset(train_df, train_tfms)val_ds = AgeGenderDataset(val_df, eval_tfms)test_ds = AgeGenderDataset(test_df, eval_tfms)train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)print(f'Train batches: {len(train_loader)}')print(f'Val batches:   {len(val_loader)}')print(f'Test batches:  {len(test_loader)}')

## 12. Model: Face Attribute Custom Head (Age + Gender)

In [ ]:
class AgeGenderNet(nn.Module):    def __init__(self):        super().__init__()        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)        in_features = backbone.fc.in_features        backbone.fc = nn.Identity()        self.backbone = backbone        self.gender_head = nn.Linear(in_features, 2)        self.age_head = nn.Linear(in_features, 1)    def forward(self, x):        feat = self.backbone(x)        gender_logits = self.gender_head(feat)        age_pred = self.age_head(feat).squeeze(1)        return gender_logits, age_predmodel = AgeGenderNet().to(DEVICE)optim = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)ce_loss = nn.CrossEntropyLoss()l1_loss = nn.L1Loss()AGE_LOSS_WEIGHT = 0.2print('Model initialized.')

## 13. Train and Validate

In [ ]:
EPOCHS = 5best_val_score = float('inf')history = []for epoch in range(EPOCHS):    model.train()    train_loss = 0.0    train_count = 0    for xb, age, gender, _ in train_loader:        xb = xb.to(DEVICE)        age = age.to(DEVICE)        gender = gender.to(DEVICE)        optim.zero_grad()        gender_logits, age_pred = model(xb)        loss_gender = ce_loss(gender_logits, gender)        loss_age = l1_loss(age_pred, age)        loss = loss_gender + AGE_LOSS_WEIGHT * loss_age        loss.backward()        optim.step()        train_loss += float(loss.item()) * xb.size(0)        train_count += xb.size(0)    model.eval()    val_loss = 0.0    val_count = 0    val_gender_true = []    val_gender_pred = []    val_age_true = []    val_age_pred = []    with torch.no_grad():        for xb, age, gender, _ in val_loader:            xb = xb.to(DEVICE)            age = age.to(DEVICE)            gender = gender.to(DEVICE)            gender_logits, age_out = model(xb)            loss_gender = ce_loss(gender_logits, gender)            loss_age = l1_loss(age_out, age)            loss = loss_gender + AGE_LOSS_WEIGHT * loss_age            val_loss += float(loss.item()) * xb.size(0)            val_count += xb.size(0)            val_gender_true.extend(gender.cpu().numpy().tolist())            val_gender_pred.extend(gender_logits.argmax(dim=1).cpu().numpy().tolist())            val_age_true.extend(age.cpu().numpy().tolist())            val_age_pred.extend(age_out.cpu().numpy().tolist())    train_loss = train_loss / max(1, train_count)    val_loss = val_loss / max(1, val_count)    val_acc = accuracy_score(val_gender_true, val_gender_pred)    val_mae = mean_absolute_error(val_age_true, val_age_pred)    score = val_loss + 0.01 * val_mae - 0.1 * val_acc    history.append({'epoch': epoch + 1, 'train_loss': train_loss, 'val_loss': val_loss, 'val_gender_acc': val_acc, 'val_age_mae': val_mae})    print(f"Epoch {epoch + 1}/{EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | val_age_mae={val_mae:.4f}")    if score < best_val_score:        best_val_score = score        torch.save(model.state_dict(), RUNS_DIR / 'best_age_gender_model.pt')print('Training complete. Best model saved.')

## 14. Test Evaluation (Real Metrics)

In [ ]:
model.load_state_dict(torch.load(RUNS_DIR / 'best_age_gender_model.pt', map_location=DEVICE))model.eval()gender_true = []gender_pred = []age_true = []age_pred = []path_list = []with torch.no_grad():    for xb, age, gender, paths in test_loader:        xb = xb.to(DEVICE)        g_logits, a_pred = model(xb)        g_hat = g_logits.argmax(dim=1).cpu().numpy().tolist()        gender_true.extend(gender.numpy().tolist())        gender_pred.extend(g_hat)        age_true.extend(age.numpy().tolist())        age_pred.extend(a_pred.cpu().numpy().tolist())        path_list.extend(list(paths))gender_acc = accuracy_score(gender_true, gender_pred)gender_f1 = f1_score(gender_true, gender_pred, average='macro')age_mae = mean_absolute_error(age_true, age_pred)age_rmse = mean_squared_error(age_true, age_pred) ** 0.5metrics = {    'dataset': KAGGLE_DATASET,    'task': 'age_gender_attributes',    'gender_accuracy': float(gender_acc),    'gender_macro_f1': float(gender_f1),    'age_mae': float(age_mae),    'age_rmse': float(age_rmse),    'test_size': int(len(path_list))}metrics_path = ARTIFACTS_DIR / 'metrics.json'metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')print(json.dumps(metrics, indent=2))print('Gender report:')print(classification_report(gender_true, gender_pred, target_names=['male(0)', 'female(1)']))

## 15. Error Analysis and Honest Qualitative Review

In [ ]:
pred_df = pd.DataFrame({    'path': path_list,    'gender_true': gender_true,    'gender_pred': gender_pred,    'age_true': age_true,    'age_pred': age_pred})pred_df['abs_age_error'] = (pred_df['age_true'] - pred_df['age_pred']).abs()pred_df['gender_correct'] = (pred_df['gender_true'] == pred_df['gender_pred'])cm = confusion_matrix(gender_true, gender_pred)fig, ax = plt.subplots(figsize=(5, 4))ax.imshow(cm, cmap='Blues')ax.set_title('Gender Confusion Matrix')ax.set_xlabel('Predicted')ax.set_ylabel('True')ax.set_xticks([0, 1])ax.set_yticks([0, 1])for i in range(2):    for j in range(2):        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='black')plt.tight_layout()cm_path = ARTIFACTS_DIR / 'gender_confusion_matrix.png'plt.savefig(cm_path, dpi=150, bbox_inches='tight')plt.show()hard_df = pred_df.sort_values('abs_age_error', ascending=False).head(6)fig, axes = plt.subplots(2, 3, figsize=(12, 8))for ax, (_, row) in zip(axes.flatten(), hard_df.iterrows()):    img = cv2.imread(row['path'])    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)    ax.imshow(img)    ax.set_title(f"age {row['age_true']:.0f}->{row['age_pred']:.1f} | g {row['gender_true']}->{row['gender_pred']}")    ax.axis('off')for ax in axes.flatten()[len(hard_df):]:    ax.axis('off')plt.tight_layout()hard_path = ARTIFACTS_DIR / 'hard_cases.png'plt.savefig(hard_path, dpi=150, bbox_inches='tight')plt.show()print(f'Saved error analysis artifacts: {cm_path}, {hard_path}')

## 16. Optional DeepFace Attribute Baseline on Sample

In [ ]:
df_sample = test_df.sample(n=min(20, len(test_df)), random_state=SEED)baseline_rows = []for _, row in df_sample.iterrows():    fp = row['path']    try:        out = DeepFace.analyze(img_path=fp, actions=['age', 'gender'], detector_backend='opencv', enforce_detection=False, silent=True)        item = out[0] if isinstance(out, list) else out        g_label = item.get('dominant_gender', 'Unknown')        g_hat = 0 if str(g_label).lower().startswith('man') else 1        baseline_rows.append({            'path': fp,            'age_true': float(row['age']),            'age_pred_deepface': float(item.get('age', np.nan)),            'gender_true': int(row['gender']),            'gender_pred_deepface': int(g_hat)        })    except Exception:        continuebaseline_df = pd.DataFrame(baseline_rows)if len(baseline_df) > 0:    deepface_gender_acc = accuracy_score(baseline_df['gender_true'], baseline_df['gender_pred_deepface'])    deepface_age_mae = mean_absolute_error(baseline_df['age_true'], baseline_df['age_pred_deepface'])    print(f'DeepFace sample size: {len(baseline_df)}')    print(f'DeepFace gender acc: {deepface_gender_acc:.4f}')    print(f'DeepFace age MAE: {deepface_age_mae:.4f}')else:    print('DeepFace sample baseline unavailable due to runtime errors or empty sample.')

## 17. Save Real Outputs and Manifest

In [ ]:
final_model_path = ARTIFACTS_DIR / 'best_age_gender_model.pt'torch.save(model.state_dict(), final_model_path)pred_path = ARTIFACTS_DIR / 'test_predictions.csv'pred_df.to_csv(pred_path, index=False)hist_path = ARTIFACTS_DIR / 'train_history.csv'pd.DataFrame(history).to_csv(hist_path, index=False)manifest = {    'project': 'Age & Gender Recognition',    'dataset': KAGGLE_DATASET,    'task_family': 'face detection + attributes',    'stack': 'DeepFace detector + custom PyTorch head',    'artifacts': {        'model_pt': str(final_model_path),        'metrics_json': str(metrics_path),        'predictions_csv': str(pred_path),        'history_csv': str(hist_path),        'eda_samples_png': str(sample_grid_path),        'eda_distributions_png': str(eda_dist_path),        'gender_confusion_png': str(cm_path),        'hard_cases_png': str(hard_path)    }}manifest_path = ARTIFACTS_DIR / 'artifact_manifest.json'manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')print(json.dumps(manifest, indent=2))

## 18. Final SummaryThis notebook performs real UTKFace download, validates files and parsed labels, verifies split quality, checks face detection behavior, trains/evaluates a custom age-gender head, and saves only generated outputs.